In [1]:
import pandas as pd
import geopandas as gpd
import unicodedata, re

import pandas as pd
import geopandas as gpd
import unicodedata, re


In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────

ORIGINS   = "data/origins.csv"
COMMUNES  = "data/limadmin-shp/LIMADM_COMMUNES.shp"
SES       = "data/indice_socio_economique_communes.csv"
CENSUS    = "data/lux_census_2021_socio.csv"
OUT       = "data/grid_to_municipality.csv"
ASSUMED_CRS = "EPSG:2169"
SES_COL = "Indice_socio_economique"


def norm(s):
# turns a commune name into a normalization key for joining the SES
#file to the spatial data. It strips accents, lowercases, and drops all non-alphanumeric
#characters, so spelling differences in accents/case/punctuation don't break the merge
#(e.g. `"Béiwen-um-Atert"` → `"beiwenumatert"`). It handles only formatting; real name
#mismatches like merged communes are resolved separately in `MERGERS`.
    
    if pd.isna(s): return ""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]", "", s.lower())


In [27]:

# 1. Origins → GeoDataFrame (only need GRD_ID + coordinates here)
o = pd.read_csv(ORIGINS)
gdf_o = gpd.GeoDataFrame(
    o[["GRD_ID", "x", "y"]],
    geometry=gpd.points_from_xy(o.x, o.y),
    crs=ASSUMED_CRS,
)

# 2. Communes
gdf_c = gpd.read_file(COMMUNES).to_crs(ASSUMED_CRS)[["COMMUNE", "geometry"]]

# 3. Spatial join: within first, then nearest for border cells
joined = gpd.sjoin(gdf_o, gdf_c, how="left", predicate="within")
unmatched_mask = joined["COMMUNE"].isna()
print(f"Within-join unmatched: {unmatched_mask.sum()}")

if unmatched_mask.any():
    fallback = gpd.sjoin_nearest(
        gdf_o[unmatched_mask].drop(columns=["index_right"], errors="ignore"),
        gdf_c, how="left", max_distance=2000
    )
    joined.loc[unmatched_mask, "COMMUNE"] = fallback["COMMUNE"].values

print(f"Still unmatched after nearest: {joined['COMMUNE'].isna().sum()}")
joined = joined.rename(columns={"COMMUNE": "municipality"})



Within-join unmatched: 201
Still unmatched after nearest: 0


In [28]:
# 4. Population from census (T = NAT + EU_OTH + OTH), same as Theil script
census = pd.read_csv(CENSUS)
census["T"] = census[["NAT", "EU_OTH", "OTH"]].sum(axis=1)
joined = joined.merge(
    census[["SPATIAL", "T"]], left_on="GRD_ID", right_on="SPATIAL", how="left"
)

# 5. SES file: handle merged communes BEFORE building keys
ses = pd.read_csv(SES)
MERGERS = {
    "Habscht":     ["Hobscheid", "Septfontaines"],
    "Helperknapp": ["Boevange-sur-Attert", "Tuntange"],
}
merge_new = []
for new_name, parents in MERGERS.items():
    vals = ses.loc[ses["Commune"].isin(parents), SES_COL]
    if len(vals) != len(parents):
        found = ses.loc[ses["Commune"].isin(parents), "Commune"].tolist()
        print(f"WARNING: {new_name} expected {parents}, found {found}")
    merge_new.append({"Commune": new_name, SES_COL: vals.mean()})
ses = pd.concat([ses, pd.DataFrame(merge_new)], ignore_index=True)

# 6. Normalize and merge
ses["_key"] = ses["Commune"].map(norm)
joined["_key"] = joined["municipality"].map(norm)

unmatched = set(joined["_key"]) - set(ses["_key"])
if unmatched:
    print(f"WARNING: {len(unmatched)} commune names didn't match SES file:")
    for u in sorted(unmatched)[:20]:
        print("  ", u)

m = joined.merge(ses[["_key", SES_COL]], on="_key", how="left")

print("done")

In [29]:
# 7. Population-weighted terciles using T
m_for_tercile = m.dropna(subset=[SES_COL]).copy()
m_for_tercile["T"] = m_for_tercile["T"].fillna(0)
m_for_tercile = m_for_tercile[m_for_tercile["T"] > 0].sort_values(SES_COL)
cum = m_for_tercile["T"].cumsum() / m_for_tercile["T"].sum()
m_for_tercile["ses_tercile"] = pd.cut(
    cum, bins=[-0.001, 1/3, 2/3, 1.001], labels=["low", "mid", "high"]
)

# Map tercile back per commune (all cells in same commune share tercile)
commune_to_tercile = (
    m_for_tercile.drop_duplicates("municipality")
    .set_index("municipality")["ses_tercile"]
)
m["ses_tercile"] = m["municipality"].map(commune_to_tercile)

print(m_for_tercile.groupby("ses_tercile", observed=True)["T"].sum()
      / m_for_tercile["T"].sum())
print(f"Cells with tercile assigned: {m['ses_tercile'].notna().sum()} / {len(m)}")



ses_tercile
low     0.330963
mid     0.335633
high    0.333404
Name: T, dtype: float64
Cells with tercile assigned: 2795 / 2795


In [30]:
# 8. Write
m[["GRD_ID", "municipality", "ses_tercile"]].to_csv(OUT, index=False)
print(f"Wrote {OUT}: {len(m)} rows")

Wrote grid_to_municipality.csv: 2795 rows


In [31]:
# mean of tectiles
print(m_for_tercile.groupby("ses_tercile", observed=True)
      .apply(lambda g: (g[SES_COL] * g["T"]).sum() / g["T"].sum()))

ses_tercile
low     0.288866
mid     0.440189
high    0.737835
dtype: float64
